In [1]:
!pip install catboost

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from catboost import CatBoostClassifier

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sample_submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

# Quick look at training data
print("Training Data Shape:", train.shape)
print("\nFirst 5 rows:")
train.head()

Training Data Shape: (594194, 21)

First 5 rows:


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
# Save ids and target
train_id = train['id']
test_id = test['id']
y = (train['Churn'] == 'Yes').astype(int)

# Drop id and target from features
train = train.drop(['id', 'Churn'], axis=1)
test = test.drop('id', axis=1)

# --- Feature Engineering (same as before) ---
def engineer_features(df):
    df = df.copy()
    # Tenure groups
    df['tenure_group'] = pd.cut(df['tenure'], bins=[0,6,12,24,48,100], labels=['0-6','6-12','12-24','24-48','48+'])
    # Avg monthly
    df['avg_monthly'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    # Binary flags for services
    service_cols = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                    'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    for col in service_cols:
        df[col + '_bin'] = (df[col] == 'Yes').astype(int)
    df['service_count'] = df[[col+'_bin' for col in service_cols]].sum(axis=1)
    df['has_internet'] = (df['InternetService'] != 'No').astype(int)
    # Polynomials
    df['tenure_sq'] = df['tenure'] ** 2
    df['monthly_sq'] = df['MonthlyCharges'] ** 2
    df['tenure_x_monthly'] = df['tenure'] * df['MonthlyCharges']
    return df

train_fe = engineer_features(train)
test_fe = engineer_features(test)

# --- Encode all categoricals (including new ones) ---
# Combine for consistent encoding
combined = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)

# Identify object columns
cat_cols = combined.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns to encode:", cat_cols)

# Label encode each
for col in cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

# Split back
train_enc = combined.iloc[:len(train_fe)].copy()
test_enc = combined.iloc[len(train_fe):].copy()

# Add target back to train_enc (optional)
train_enc['Churn'] = y.values

# Verify all columns are numeric
print("\nData types after encoding:")
print(train_enc.dtypes.value_counts())

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Data types after encoding:
int64       29
float64      5
category     1
Name: count, dtype: int64


In [6]:
# Combine train_fe and test_fe
combined = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)

# Encode ALL categorical columns in combined (both object and category dtypes)
# First, handle object columns with LabelEncoder

cat_cols = combined.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

# Convert any pandas 'category' columns to integer codes (e.g., tenure_group)
cat_dtype_cols = combined.select_dtypes(include=['category']).columns.tolist()
for col in cat_dtype_cols:
    combined[col] = combined[col].cat.codes

#  Now split back into train_enc and test_enc
train_enc = combined.iloc[:len(train_fe)].copy()
test_enc = combined.iloc[len(train_fe):].copy()

# Add target back to train_enc (using y from original train)
# (Make sure y is defined; if not, reload: original_train = pd.read_csv(...); y = (original_train['Churn'] == 'Yes').astype(int))
train_enc['Churn'] = y.values

# Verify all columns are numeric
print("Data types after encoding:")
print(train_enc.dtypes.value_counts())
print("\nAny missing values?", train_enc.isnull().any().any())

Data types after encoding:
int64      29
float64     5
int8        1
Name: count, dtype: int64

Any missing values? False


In [7]:
feature_cols = [col for col in train_enc.columns if col != 'Churn']
X = train_enc[feature_cols].copy()
print("X shape:", X.shape)
y = train_enc['Churn']
print("y shape:", y.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X shape: (594194, 34)
y shape: (594194,)


In [8]:
# Assuming X (features) and y are already defined from your previous steps
# Make a copy to avoid modifying original
X_interact = X.copy()

# Contract is already encoded as 0,1,2 (likely 0=month-to-month, 1=one year, 2=two year)
# Create interactions with top continuous features
X_interact['Contract_x_tenure'] = X_interact['Contract'] * X_interact['tenure']
X_interact['Contract_x_MonthlyCharges'] = X_interact['Contract'] * X_interact['MonthlyCharges']
X_interact['Contract_x_TotalCharges'] = X_interact['Contract'] * X_interact['TotalCharges']

# Also with important binary features (OnlineSecurity, TechSupport, etc.)
X_interact['Contract_x_OnlineSecurity'] = X_interact['Contract'] * X_interact['OnlineSecurity']
X_interact['Contract_x_TechSupport'] = X_interact['Contract'] * X_interact['TechSupport']

# Optional: interactions among top continuous features
X_interact['tenure_x_MonthlyCharges'] = X_interact['tenure'] * X_interact['MonthlyCharges']  # already have tenure_x_monthly? might be duplicate

print("New features added. Shape:", X_interact.shape)

New features added. Shape: (594194, 40)


In [9]:
test_interact = test_enc.copy()

# Add interaction features (match what you did for X_interact)
test_interact['Contract_x_tenure'] = test_interact['Contract'] * test_interact['tenure']
test_interact['Contract_x_MonthlyCharges'] = test_interact['Contract'] * test_interact['MonthlyCharges']
test_interact['Contract_x_TotalCharges'] = test_interact['Contract'] * test_interact['TotalCharges']
test_interact['Contract_x_OnlineSecurity'] = test_interact['Contract'] * test_interact['OnlineSecurity']
test_interact['Contract_x_TechSupport'] = test_interact['Contract'] * test_interact['TechSupport']
test_interact['tenure_x_MonthlyCharges'] = test_interact['tenure'] * test_interact['MonthlyCharges']

# Verify columns now match
print("X_interact columns:", X_interact.columns.tolist())
print("\ntest_interact columns:", test_interact.columns.tolist())
assert all(X_interact.columns == test_interact.columns), "Columns still don't match!"

X_interact columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'tenure_group', 'avg_monthly', 'PhoneService_bin', 'MultipleLines_bin', 'OnlineSecurity_bin', 'OnlineBackup_bin', 'DeviceProtection_bin', 'TechSupport_bin', 'StreamingTV_bin', 'StreamingMovies_bin', 'service_count', 'has_internet', 'tenure_sq', 'monthly_sq', 'tenure_x_monthly', 'Contract_x_tenure', 'Contract_x_MonthlyCharges', 'Contract_x_TotalCharges', 'Contract_x_OnlineSecurity', 'Contract_x_TechSupport', 'tenure_x_MonthlyCharges']

test_interact columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',

In [32]:
param_dist_cat = {
    'iterations': [500, 1000],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'border_count': [32, 64, 128]
}

In [19]:
cat_model = CatBoostClassifier(random_seed=42, verbose=0)

random_search_cat = RandomizedSearchCV(
    cat_model,
    param_distributions=param_dist_cat,
    n_iter=20,                 # fewer iterations for speed
    cv=skf,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_cat.fit(X_interact, y)

print("Best CatBoost CV AUC:", random_search_cat.best_score_)
print("Best params:", random_search_cat.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CatBoost CV AUC: 0.9162142516992352
Best params: {'learning_rate': 0.1, 'l2_leaf_reg': 3, 'iterations': 500, 'depth': 6, 'border_count': 128}


In [33]:
# Ensure test_interact has same columns as X_interact (it should)
assert all(X_interact.columns == test_interact.columns), "Column mismatch!"

# Load original test.csv to get the IDs
test_original = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
best_cat = CatBoostClassifier(**random_search_cat.best_params_, random_seed=42, verbose=0)
best_cat.fit(X_interact, y)
cat_preds = best_cat.predict_proba(test_interact)[:, 1]
cat_sub = pd.DataFrame({'id': test_original['id'], 'Churn': cat_preds})
cat_sub.to_csv('catboost_submission.csv', index=False)

In [35]:
from IPython.display import FileLink
FileLink('catboost_submission.csv')

/kaggle/working/catboost_submission.csv